# 0. Handle Missing Values & Feature Engineering

This notebook handles duplicate rows, checks for missing values, performs feature engineering on the raw credit card fraud detection dataset, and outputs a cleaned version of the dataset ready for outlier handling.

## 1. Environment Setup & Imports

Import required Python libraries for data manipulation, mathematical operations, and environment configuration.

In [1]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
print("Setup complete.")

Setup complete.


## 2. Load Raw Dataset

Read the raw credit card transaction training dataset `fraudTrain.csv` into a pandas DataFrame.

In [2]:
# Load the raw dataset
df = pd.read_csv('../../dataset/raw/fraudTrain.csv')
print(f"Loaded raw dataset with shape: {df.shape}")

Loaded raw dataset with shape: (1296675, 23)


## 3. Clean Duplicates & Inspect Missing Values

Perform checks for duplicated rows and identify missing values per column. Deduplicate any duplicate rows.

In [3]:
# Clean duplicates and check nulls
initial_len = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {initial_len - len(df)} duplicate rows.")
print("Missing values per column:")
print(df.isnull().sum())

Dropped 0 duplicate rows.
Missing values per column:
Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
merch_lat                0
merch_long               0
is_fraud                 0
dtype: int64


## 4. Helper Functions for Feature Engineering

Define mathematical formulas and search algorithms for feature calculations:
- `haversine_np`: Computes the geodesic distance in kilometers between customer and merchant.
- `compute_velocity_and_freq`: Uses a highly optimized vectorized search (`numpy.searchsorted`) to count transaction velocity in the last 24 hours and track historical transaction frequencies without data leakage.

In [4]:
# Helper function for vectorized Haversine distance computation
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

# Helper function to compute velocity features quickly (under 2 seconds for 1.3M rows)
def compute_velocity_and_freq(df):
    df_sorted = df.sort_values(by=['cc_num', 'unix_time']).reset_index(drop=True)
    
    velocities = np.zeros(len(df_sorted), dtype=int)
    frequencies = np.zeros(len(df_sorted), dtype=int)
    
    # Group by customer credit card number
    grouped = df_sorted.groupby('cc_num')
    
    for cc, group in grouped:
        times = group['unix_time'].values
        indices = group.index.values
        
        # searchsorted returns the first index where time >= current_time - 24 hours (86400 seconds)
        start_indices = np.searchsorted(times, times - 86400, side='left')
        local_indices = np.arange(len(group))
        
        velocities[indices] = local_indices - start_indices
        frequencies[indices] = local_indices
        
    df_sorted['velocity_last_24h'] = velocities
    df_sorted['transaction_frequency'] = frequencies
    return df_sorted

## 5. Feature Engineering

Apply transformations to create engineered features from raw values:
- Extract datetime and time components (hour, day of week, weekend indicator, month).
- Calculate customer age.
- Compute merchant distance and flag location mismatch.
- Calculate amount Z-scores and log-transformed amounts.
- Generate customer velocities, transaction frequencies, foreign transaction flags, and night transaction indicators.

In [5]:
# Feature Engineering
print("Engineering features...")

# 1. Datetime conversions
df['transaction_datetime'] = pd.to_datetime(df['trans_date_trans_time'])

# 2. Age from Date of Birth
df['dob'] = pd.to_datetime(df['dob'])
df['customer_age'] = (df['transaction_datetime'] - df['dob']).dt.days // 365

# 3. Time features
df['transaction_hour'] = df['transaction_datetime'].dt.hour
df['day_of_week'] = df['transaction_datetime'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['transaction_month'] = df['transaction_datetime'].dt.month

# 4. Distance features
df['distance_to_merchant'] = haversine_np(df['lat'], df['long'], df['merch_lat'], df['merch_long'])
df['location_mismatch'] = (df['distance_to_merchant'] > 80).astype(int)

# 5. Amount features
df['amount_zscore'] = (df['amt'] - df['amt'].mean()) / df['amt'].std()
df['amount_log'] = np.log1p(df['amt'])

# 6. Night transaction (10 PM to 6 AM)
df['night_transaction'] = df['transaction_hour'].apply(lambda x: 1 if x >= 22 or x <= 6 else 0)

# 7. Velocity and frequency features
df = compute_velocity_and_freq(df)

# 8. Foreign transaction (State/location comparison estimate)
df['foreign_transaction'] = (df['distance_to_merchant'] > 150).astype(int)

print("Feature engineering completed.")

Engineering features...
Feature engineering completed.


## 6. Rename and Filter Columns

Rename the raw columns to standard names and filter the final dataset to match the recommended 13 columns for the model.

In [6]:
# Rename and select columns to match the recommended final dataset
df = df.rename(columns={
    'unix_time': 'transaction_unix_time',
    'amt': 'amount',
    'category': 'merchant_category',
    'city': 'customer_city',
    'state': 'customer_state',
    'lat': 'customer_latitude',
    'long': 'customer_longitude',
    'city_pop': 'city_population',
    'job': 'occupation',
    'merch_lat': 'merchant_latitude',
    'merch_long': 'merchant_longitude'
})

recommended_cols = [
    'distance_to_merchant',
    'customer_age',
    'transaction_hour',
    'day_of_week',
    'is_weekend',
    'location_mismatch',
    'velocity_last_24h',
    'foreign_transaction',
    'amount',
    'merchant_category',
    'city_population',
    'gender',
    'is_fraud'
]

df_final = df[recommended_cols]
print(f"Final dataset shape: {df_final.shape}")
print(f"Final columns: {list(df_final.columns)}")

Final dataset shape: (1296675, 13)
Final columns: ['distance_to_merchant', 'customer_age', 'transaction_hour', 'day_of_week', 'is_weekend', 'location_mismatch', 'velocity_last_24h', 'foreign_transaction', 'amount', 'merchant_category', 'city_population', 'gender', 'is_fraud']


## 7. Save Processed Dataset

Save the final processed dataset to `dataset/processed/credit_card_fraud_null_handled.csv`.

In [7]:
# Save output processed data
output_path = '../../dataset/processed/credit_card_fraud_null_handled.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_final.to_csv(output_path, index=False)
print(f"Saved null-handled and feature engineered dataset to: {output_path}")

Saved null-handled and feature engineered dataset to: ../../dataset/processed/credit_card_fraud_null_handled.csv
